# 🎯 Phase 3a: Placement Prediction
Binary Classification — Placed vs Not Placed

In [ ]:
import pandas as pd, numpy as np, warnings, pickle, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

X_train = pd.read_csv('data/X_train_placement.csv')
X_test  = pd.read_csv('data/X_test_placement.csv')
y_train = pd.read_csv('data/y_train_placement.csv').squeeze()
y_test  = pd.read_csv('data/y_test_placement.csv').squeeze()
print("✅ Data loaded | Train:", X_train.shape, "Test:", X_test.shape)


## 🤖 Train 4 Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=200, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    acc  = accuracy_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    results[name] = {'model': model, 'acc': acc, 'auc': auc, 'pred': y_pred, 'prob': y_prob}
    print(f"{name:25s} | Accuracy: {acc:.4f} | AUC: {auc:.4f}")

best_name = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_name]['model']
print(f"\n🏆 Best Model: {best_name}")


## 📊 Evaluation — Confusion Matrix & ROC

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix of best model
cm = confusion_matrix(y_test, results[best_name]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Not Placed','Placed'], yticklabels=['Not Placed','Placed'])
axes[0].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

# ROC curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC Curves Comparison', fontweight='bold')
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(fontsize=8)

# Accuracy comparison
names = list(results.keys())
accs  = [results[n]['acc'] for n in names]
colors = ['#2ecc71' if n==best_name else '#3498db' for n in names]
bars = axes[2].bar(names, accs, color=colors, edgecolor='white')
for bar, acc in zip(bars, accs):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                 f'{acc:.3f}', ha='center', fontweight='bold', fontsize=10)
axes[2].set_ylim(0.75, 1.0)
axes[2].set_title('Model Accuracy Comparison', fontweight='bold')
axes[2].set_ylabel('Accuracy')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/07_placement_model.png', bbox_inches='tight')
plt.show()


## 🔍 Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns)
    imp = imp.sort_values(ascending=True).tail(15)
    plt.figure(figsize=(10, 7))
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(imp)))
    plt.barh(imp.index, imp.values, color=colors)
    plt.title(f'Top 15 Feature Importances\n({best_name})', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('outputs/08_placement_features.png', bbox_inches='tight')
    plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, results[best_name]['pred'],
      target_names=['Not Placed','Placed']))


## 💾 Save Best Model

In [ ]:
os.makedirs('models', exist_ok=True)
with open('models/placement_model.pkl','wb') as f:
    pickle.dump(best_model, f)
print(f"✅ Saved models/placement_model.pkl  ({best_name})")
print(f"   Accuracy: {results[best_name]['acc']:.4f} | AUC: {results[best_name]['auc']:.4f}")
